   # Insurance Claims Analytics & Fraud Risk Analysis
   Analysing a synthetic motor insurance claims dataset to understand claims volume, processing performance (SLA), and patterns associated with potentially fraudulent claims.

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/insurance_claims.csv")
df.shape

(15140, 24)

In [4]:
df.head()

,claim_id,policy_id,claim_date,incident_date,claim_type,claim_status,customer_age,customer_segment,policy_tenure_months,vehicle_age,...,settlement_amount,processing_days,documents_submitted,documents_required,previous_claims,fraud_flag,severity,handler_experience_years,customer_satisfaction,investigation_required
0,CLM003141,POL004427,2025-06-16,2025-06-09,Third-party,Pending Documents,37.0,Standard,62,2,...,0.0,9,3,5,0,0,High,14,NaN,0
1,CLM000029,POL004909,2024-10-08,2024-10-05,Collision,Closed/Rejected,39.0,Standard,25,1,...,0.0,10,4,4,0,0,Low,3,3.0,0
2,CLM009945,POL001520,2025-08-14,2025-08-14,Collision,Under Investigation,49.0,Standard,12,1,...,0.0,21,4,4,1,0,Medium,1,4.0,1
3,CLM009858,POL006450,2025-01-19,2025-01-19,Third-party,Pending Documents,28.0,Standard,3,5,...,0.0,12,3,5,0,0,Medium,3,4.0,0
4,CLM010021,POL001826,2024-12-10,2024-12-07,Windscreen,Settled,37.0,Standard,66,8,...,5000.0,1,2,2,1,0,Medium,2,4.0,0


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15140 entries, 0 to 15139
Data columns (total 24 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   claim_id                  15140 non-null  str    
 1   policy_id                 15140 non-null  str    
 2   claim_date                15140 non-null  str    
 3   incident_date             15140 non-null  str    
 4   claim_type                15140 non-null  str    
 5   claim_status              15140 non-null  str    
 6   customer_age              14918 non-null  float64
 7   customer_segment          15140 non-null  str    
 8   policy_tenure_months      15140 non-null  int64  
 9   vehicle_age               15140 non-null  int64  
 10  vehicle_type              15140 non-null  str    
 11  region                    15140 non-null  str    
 12  channel                   15140 non-null  str    
 13  claim_amount              15140 non-null  float64
 14  settlement_amount

In [6]:
df.describe()

,customer_age,policy_tenure_months,vehicle_age,claim_amount,settlement_amount,processing_days,documents_submitted,documents_required,previous_claims,fraud_flag,handler_experience_years,customer_satisfaction,investigation_required
count,14918.000000,15140.000000,15140.000000,1.514000e+04,1.514000e+04,15140.000000,15140.000000,15140.000000,15140.000000,15140.000000,15140.000000,13562.000000,15140.000000
mean,40.003352,36.040621,4.959247,1.689956e+05,7.736750e+04,9.307001,3.643527,3.936328,0.812153,0.056341,4.422919,3.592980,0.137384
std,11.853336,25.464229,3.402651,2.939033e+05,1.853758e+05,8.233469,1.302493,1.159532,0.904801,0.230586,3.122687,0.950638,0.344264
min,0.000000,1.000000,0.000000,-2.591000e+05,0.000000e+00,1.000000,0.000000,2.000000,0.000000,0.000000,0.000000,1.000000,0.000000
25%,32.000000,17.000000,2.000000,3.110000e+04,0.000000e+00,4.000000,3.000000,3.000000,0.000000,0.000000,2.000000,3.000000,0.000000
50%,40.000000,30.000000,4.000000,7.180000e+04,1.330000e+04,7.000000,4.000000,4.000000,1.000000,0.000000,4.000000,4.000000,0.000000
75%,48.000000,48.000000,7.000000,1.711250e+05,7.220000e+04,11.000000,5.000000,5.000000,1.000000,0.000000,6.000000,4.000000,0.000000
max,212.000000,180.000000,15.000000,5.457700e+06,5.124000e+06,86.000000,6.000000,6.000000,6.000000,1.000000,20.000000,5.000000,1.000000


In [7]:
df.isnull().sum()

claim_id                       0
policy_id                      0
claim_date                     0
incident_date                  0
claim_type                     0
claim_status                   0
customer_age                 222
customer_segment               0
policy_tenure_months           0
vehicle_age                    0
vehicle_type                   0
region                         0
channel                        0
claim_amount                   0
settlement_amount              0
processing_days                0
documents_submitted            0
documents_required             0
previous_claims                0
fraud_flag                     0
severity                       0
handler_experience_years       0
customer_satisfaction       1578
investigation_required         0
dtype: int64

In [8]:
df["region"].value_counts()

region
North        3522
South        3276
West         3162
East         2727
Central      2220
north          47
south          30
west           28
east           26
 North         22
central        21
 South         19
 West          15
 East          13
 Central       12
Name: count, dtype: int64

In [9]:
df["channel"].value_counts()

channel
Online        4188
Phone         3325
Mobile App    3046
Broker        2674
Branch        1846
mobile app      61
Name: count, dtype: int64

In [10]:
df["claim_type"].value_counts()

claim_type
Collision            5255
Third-party          2785
Accidental damage    2535
Windscreen           2310
Theft                1189
Fire                 1026
Third Party            40
Name: count, dtype: int64

In [11]:
(pd.to_datetime(df["claim_date"]) < pd.to_datetime(df["incident_date"])).sum()

np.int64(12)

## Data quality issues found

## Data quality issues found

The raw dataset contains 15,140 rows and 24 columns. Exploration revealed the following issues:

1. **Duplicate rows:** 140 exact duplicate records.
2. **Missing customer age:** 222 records have no age recorded.
3. **Impossible customer ages:** 6 records have ages outside a realistic range (e.g. 0, -3, 150, 212).
4. **Negative claim amounts:** 5 claims have a negative value, which is not possible and suggests data entry errors.
5. **Inconsistent region labels:** 233 records use non-standard formatting: 152 in lowercase (e.g. "north") and 81 with extra spaces (e.g. " North ").
6. **Inconsistent channel labels:** 61 records use "mobile app" instead of "Mobile App".
7. **Inconsistent claim type labels:** 40 records use "Third Party" instead of "Third-party".
8. **Invalid date logic:** 12 claims have a claim date earlier than the incident date, which is not possible.

**Other observations**
- `claim_date` and `incident_date` are stored as text and need converting to dates.
- `customer_satisfaction` has 1,578 missing values. These represent customers who did not complete the satisfaction survey, so they will be **kept as missing** rather than filled in, to avoid biasing the satisfaction analysis.

In [12]:
print("Rows before:", len(df))
df = df.drop_duplicates()
print("Rows after:", len(df))

Rows before: 15140
Rows after: 15000


### Fix 2: Standardise text labels
Inconsistent casing, extra spaces and naming variations would split one category into several in charts and SQL GROUP BYs.


In [14]:
df["region"] = df["region"].str.strip().str.title()
df["channel"] = df["channel"].str.strip().str.title()
df["claim_type"] = df["claim_type"].str.strip().replace({"Third Party": "Third-party"})

print("Regions:", df["region"].nunique())
print("Channels:", df["channel"].nunique())
print("Claim types:", df["claim_type"].nunique())

Regions: 5
Channels: 5
Claim types: 6


### Fix 3: Convert dates and remove impossible records
Dates are converted from text to datetime. Claims dated before the incident are logically impossible, so they are removed.

In [15]:
df["claim_date"] = pd.to_datetime(df["claim_date"])
df["incident_date"] = pd.to_datetime(df["incident_date"])

bad_dates = df["claim_date"] < df["incident_date"]
print("Claims before incident:", bad_dates.sum())
df = df[~bad_dates]

Claims before incident: 12


### Fix 4: Handle invalid and missing ages
Ages below 18 or above 100 are treated as invalid and set to missing. All missing ages are then filled with the median age, which is robust to outliers.

In [16]:
df.loc[(df["customer_age"] < 18) | (df["customer_age"] > 100), "customer_age"] = np.nan
print("Missing ages to fill:", df["customer_age"].isnull().sum())

median_age = df["customer_age"].median()
df["customer_age"] = df["customer_age"].fillna(median_age)
print("Median age used:", median_age)

Missing ages to fill: 225
Median age used: 40.0


### Fix 5: Remove negative claim amounts
A claim amount cannot be negative. These are treated as data entry errors and removed.

In [17]:
print("Negative amounts:", (df["claim_amount"] <= 0).sum())
df = df[df["claim_amount"] > 0]
print("Final rows:", len(df))

Negative amounts: 5
Final rows: 14983


In [18]:
df.info()

<class 'pandas.DataFrame'>
Index: 14983 entries, 0 to 15139
Data columns (total 24 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   claim_id                  14983 non-null  str           
 1   policy_id                 14983 non-null  str           
 2   claim_date                14983 non-null  datetime64[us]
 3   incident_date             14983 non-null  datetime64[us]
 4   claim_type                14983 non-null  str           
 5   claim_status              14983 non-null  str           
 6   customer_age              14983 non-null  float64       
 7   customer_segment          14983 non-null  str           
 8   policy_tenure_months      14983 non-null  int64         
 9   vehicle_age               14983 non-null  int64         
 10  vehicle_type              14983 non-null  str           
 11  region                    14983 non-null  str           
 12  channel                   14983 no

## Feature engineering
New business fields are created to support SLA, settlement, documentation and risk analysis. The company's processing target (SLA) is assumed to be 7 days.

In [19]:
df["settlement_ratio"] = df["settlement_amount"] / df["claim_amount"]
df["sla_status"] = np.where(df["processing_days"] <= 7, "Within SLA", "Outside SLA")
df["doc_status"] = np.where(df["documents_submitted"] >= df["documents_required"], "Complete", "Incomplete")
df["value_band"] = np.where(df["claim_amount"] >= 200000, "High Value", "Standard")
df["reporting_lag_days"] = (df["claim_date"] - df["incident_date"]).dt.days
df["claim_month"] = df["claim_date"].dt.to_period("M").astype(str)

df["sla_status"].value_counts()

sla_status
Within SLA     8448
Outside SLA    6535
Name: count, dtype: int64

In [20]:
df["previous_claims"].value_counts().sort_index()

previous_claims
0    6684
1    5361
2    2210
3     571
4     135
5      19
6       3
Name: count, dtype: int64

Only ~22 customers have 5+ previous claims, so a threshold of 3+ is used in the risk score to capture a meaningful group of repeat claimants.

### Fraud-risk score
An exploratory, rule-based risk score combining several claim characteristics. This is not a production fraud model; it is used to identify patterns associated with potentially suspicious claims.

| Rule | Points |
|---|---|
| Claim amount ≥ ₹200,000 | +2 |
| 3+ previous claims | +2 |
| Investigation required | +2 |
| Processing > 14 days | +1 |
| Incomplete documents | +1 |
| High/Critical severity | +1 |

Risk level: 0–2 = Low, 3–5 = Medium, 6+ = High

In [21]:
df["risk_score"] = (
    np.where(df["claim_amount"] >= 200000, 2, 0)
    + np.where(df["previous_claims"] >= 3, 2, 0)
    + np.where(df["investigation_required"] == 1, 2, 0)
    + np.where(df["processing_days"] > 14, 1, 0)
    + np.where(df["doc_status"] == "Incomplete", 1, 0)
    + np.where(df["severity"].isin(["High", "Critical"]), 1, 0)
)

df["risk_level"] = pd.cut(df["risk_score"], bins=[-1, 2, 5, 9],
                          labels=["Low", "Medium", "High"])
df["risk_level"].value_counts()

risk_level
Low       10587
Medium     3732
High        664
Name: count, dtype: int64

In [22]:
summary = df.groupby("risk_level", observed=True).agg(
    claims=("claim_id", "count"),
    fraud_rate=("fraud_flag", "mean"),
    total_value=("claim_amount", "sum"),
)
summary["share_of_claims"] = summary["claims"] / summary["claims"].sum()
summary["share_of_value"] = summary["total_value"] / summary["total_value"].sum()
summary.round(3)

,claims,fraud_rate,total_value,share_of_claims,share_of_value
risk_level,,,,,
Low,10587,0.011,7.537014e+08,0.707,0.297
Medium,3732,0.095,1.317873e+09,0.249,0.520
High,664,0.551,4.627369e+08,0.044,0.183


**Insight:** High-risk claims make up only ~4.4% of claim volume but ~18% of total claim value, and their fraud rate (~55%) is roughly 50x that of low-risk claims. This suggests the rule-based score is effective at prioritising claims for investigation.

**Insight:** High-risk claims make up only ~4.4% of claim volume but ~18% of total claim value, and their fraud rate (~55%) is roughly 50x that of low-risk claims. This suggests the rule-based score is effective at prioritising claims for investigation.

In [23]:
df.to_csv("data/insurance_claims_clean.csv", index=False)